In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
# 처음이면 clone, 이미 있으면 pull
!git clone https://github.com/dldusgh318/mask-aware-inpainting.git
# 또는
!git checkout yeonholee

## 데이터셋 구성

In [ ]:
!python3 prepare_fixed_testset.py

In [ ]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

# 설치 및 인증
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jessicali9530/celeba-dataset -p ./mask-aware-inpainting/data/
!cd mask-aware-inpainting/data && unzip celeba-dataset.zip -d celeba_temp

In [ ]:
import os, shutil

base = './mask-aware-inpainting/data'
os.makedirs(f'{base}/celeba/img_align_celeba', exist_ok=True)

# 이미지 이동
!mv {base}/celeba_temp/img_align_celeba/* {base}/celeba/img_align_celeba/

# 메타 파일 이동 (list_eval_partition.txt 등)
!mv {base}/celeba_temp/*.txt {base}/celeba/
!mv {base}/celeba_temp/*.csv {base}/celeba/ 2>/dev/null || true

In [ ]:
!ls ./mask-aware-inpainting/data/celeba/

## 학습

In [ ]:
!python3 train.py

!python evaluate.py checkpoints/gated/best_model.pth

## 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

for model in ['vanilla', 'pconv', 'gated']:
    df = pd.read_csv(f'mask-aware-inpainting/checkpoints/{model}/history.csv')
    plt.plot(df['epoch'], df['val_psnr'], label=model)

plt.xlabel('Epoch'); plt.ylabel('Val PSNR'); plt.legend(); plt.show()